# Geração de Dados

In [257]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd
import lightgbm as lgb
import holidays
from scipy.stats import nbinom
from scipy.optimize import minimize

In [258]:

def generate_m5_simulated_data(n_stores=3, n_skus=5, start_date="2023-01-01", days=365, seed=42):
    """
    Gera um dataset simulando a dinâmica da competição M5 da Walmart:
    - Vendas de contagem (Poisson/Binomial Negativa) com zeros (intermitência).
    - Variações de preço com elasticidade-preço.
    - Sazonalidade semanal e eventos especiais.
    """
    np.random.seed(seed)
    dates = pd.date_range(start=start_date, periods=days, freq="D")
    n_days = len(dates)
    
    data_list = []

    for store_id in range(1, n_stores + 1):
        store_name = f"STORE_{store_id:02d}"
        for sku_id in range(1, n_skus + 1):
            item_id = f"FOODS_1_{sku_id:03d}"
            
            # Demanda base do SKU na loja
            base_demand = np.random.uniform(0.5, 5.0)
            
            # Fator de sazonalidade semanal (fds vende mais)
            dow_factor = np.tile([0.8, 0.85, 0.9, 0.95, 1.1, 1.4, 1.3], int(np.ceil(n_days/7)))[:n_days]
            
            # Variação e promoção de preço
            base_price = np.random.uniform(2.0, 15.0)
            prices = base_price * np.random.choice([1.0, 0.85, 0.70], size=n_days, p=[0.8, 0.15, 0.05])
            price_elasticity = np.exp(-0.15 * (prices - base_price))
            
            # Dias com eventos/promoções especiais
            event_indices = np.random.choice(n_days, size=12, replace=False)
            event_impact = np.ones(n_days)
            event_impact[event_indices] = np.random.uniform(1.3, 2.2, size=12)
            
            # Parâmetro lambda da demanda diária
            lambda_t = base_demand * dow_factor * price_elasticity * event_impact
            
            # Vendas reais observadas (intermitentes)
            sales = np.random.poisson(lambda_t)
            
            for d_idx, d_date in enumerate(dates):
                data_list.append({
                    'date': d_date,
                    'store_id': store_name,
                    'item_id': item_id,
                    'sales': sales[d_idx],
                    'sell_price': round(prices[d_idx], 2),
                    'is_event': 1 if d_idx in event_indices else 0,
                })

    return pd.DataFrame(data_list)

df_raw = generate_m5_simulated_data(n_stores=3, n_skus=5, days=365) # Função criada anteriormente

df_raw['unique_id'] = df_raw['store_id'] + '_' + df_raw['item_id']

df_nixtla = df_raw.rename(columns={
    'date': 'ds',
    'sales': 'y'
})

In [298]:
import numpy as np
import pandas as pd
from scipy.stats import nbinom
from scipy.optimize import minimize
from mlforecast import MLForecast

class ISSMForecast:
    """
    Componente de Treinamento e Inferência do ISSM (Intermittent State-Space Model)
    alinhado à visão de estoque da Lokad (foco em caudas e intermitência).
    """
    def __init__(self, fcst: MLForecast):
        self.fcst = fcst
        self.r_dict = {}
        self.r_fallback = 1.0  # Fallback para SKUs novos
    
    @property
    def model(self):
        """Retorna o modelo do MLForecast garantindo unicidade e estado válido."""
        if not hasattr(self.fcst, "models_") or not self.fcst.models_:
            raise ValueError("O objeto MLForecast ainda não foi ajustado (fit não executado).")
        
        if len(self.fcst.models_) > 1:
            raise ValueError(
                f"O ISSMForecast suporta apenas 1 modelo por vez, mas encontrou: {list(self.fcst.models_.keys())}"
            )
        
        return next(iter(self.fcst.models_.values()))
    
    def _nbinom_log_likelihood(self, params, y, lambda_t):
        """Função de verossimilhança com proteção contra underflow/instabilidade."""
        r = params[0]
        if r <= 0: 
            return 1e10
        
        lambda_t = np.maximum(lambda_t, 1e-6)
        p = r / (r + lambda_t)
        
        log_pdf = nbinom.logpmf(y, r, p)
        log_pdf = np.nan_to_num(log_pdf, neginf=-1e2) 
        
        return -np.sum(log_pdf)

    def fit(self, df_train, id_col='unique_id', time_col='ds', target_col='y', static_features=None):
        self.id_col = id_col
        self.time_col = time_col
        self.target_col = target_col
        static_features = static_features or []

        self.fcst.fit(
            df_train, 
            id_col=id_col, 
            time_col=time_col, 
            target_col=target_col, 
            static_features=static_features
        )

        df_prep = self.fcst.preprocess(
            df_train, 
            id_col=id_col, 
            time_col=time_col, 
            target_col=target_col, 
            static_features=static_features
        )

        self.exog_cols_ = self.fcst.ts.features_order_
        X_train = df_prep[self.exog_cols_]
        df_prep['lambda_t'] = self.model.predict(X_train)

        self.r_dict = {}
        for uid, group in df_prep.groupby(id_col):
            y_obs = group[target_col].values
            lambdas = group['lambda_t'].values

            res = minimize(
                self._nbinom_log_likelihood, 
                x0=[1.0], 
                method='L-BFGS-B',
                args=(y_obs, lambdas), 
                bounds=[(1e-3, 50.0)]
            )
            self.r_dict[uid] = res.x[0]

        self.r_fallback = float(np.median(list(self.r_dict.values())))

        return self

    def predict(self, h, X_df=None, quantis=[0.50, 0.67, 0.95, 0.99]):
        X_df_clean = X_df.copy() if X_df is not None else None
        if X_df_clean is not None and self.target_col in X_df_clean.columns:
            X_df_clean = X_df_clean.drop(columns=[self.target_col])

        df_pred = self.fcst.predict(h=h, X_df=X_df_clean)
        
        model_key = next(iter(self.fcst.models_.keys()))
        df_pred = df_pred.rename(columns={model_key: 'lambda_t'})
        
        df_pred['r_dispersion'] = df_pred[self.id_col].map(self.r_dict).fillna(self.r_fallback)

        for q in sorted(quantis):
            col_name = f'q_{int(q * 100)}'
            p_param = df_pred['r_dispersion'] / (df_pred['r_dispersion'] + df_pred['lambda_t'])
            df_pred[col_name] = np.ceil(nbinom.ppf(q, df_pred['r_dispersion'], p_param))

        return df_pred

In [299]:
def temporal_split_by_horizon(df, time_col='ds', horizon_days=28):
    """
    Separa estritamente o DataFrame por data de corte.
    Os últimos 'horizon_days' viram o conjunto de TESTE/VALIDAÇÃO.
    """
    df = df.sort_values(time_col)
    max_date = df[time_col].max()
    cutoff_date = max_date - pd.Timedelta(days=horizon_days)
    
    df_train = df[df[time_col] <= cutoff_date].copy()
    df_test = df[df[time_col] > cutoff_date].copy()
    
    return df_train, df_test, cutoff_date

In [300]:
us_holidays = holidays.US(years=range(2022, 2027))

_extended_holiday_dates = set()
for h_date in us_holidays.keys():
    h_timestamp = pd.Timestamp(h_date)
    for offset in range(-2, 1):  # Véspera (-2, -1) e o próprio dia (0)
        _extended_holiday_dates.add((h_timestamp + pd.Timedelta(days=offset)).date())


def is_holiday_window(dates) -> pd.Series:
    """Retorna 1 para o dia do feriado e os 2 dias que o antecedem."""
    dates_series = pd.Series(dates)
    return dates_series.dt.date.isin(_extended_holiday_dates).astype(int)

df_nixtla['is_holiday_window'] = is_holiday_window(df_nixtla['ds'])

fcst = MLForecast(
    models={
        'lgb_issm': lgb.LGBMRegressor(
            objective='poisson',
            metric='rmse',
            n_estimators=100,
            learning_rate=0.05,
            random_state=42,
            verbosity=-1
        )
    },
    freq='D',
    lags=[7, 14, 28],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    date_features=['dayofweek', 'month', 'dayofyear']
)

In [301]:
df_train, df_test, cutoff = temporal_split_by_horizon(df_nixtla, horizon_days=300)

issm = ISSMForecast(fcst)
issm.fit(df_train[["ds", "y", "sell_price", "is_event", "is_holiday_window", "unique_id"]])

In [306]:
df_estruturado = issm.predict(h=300, X_df=df_test[["ds", "sell_price", "is_event", "is_holiday_window", "unique_id"]])

In [303]:
df_estruturado.loc[:, "y"] = df_test["y"].values

In [307]:
df_estruturado

,unique_id,ds,lambda_t,r_dispersion,q_50,q_67,q_95,q_99
0,STORE_01_FOODS_1_001,2023-03-07,2.990244,50.0,3.0,4.0,6.0,8.0
1,STORE_01_FOODS_1_001,2023-03-08,3.063599,50.0,3.0,4.0,6.0,8.0
2,STORE_01_FOODS_1_001,2023-03-09,3.796481,50.0,4.0,5.0,7.0,9.0
3,STORE_01_FOODS_1_001,2023-03-10,4.222098,50.0,4.0,5.0,8.0,10.0
4,STORE_01_FOODS_1_001,2023-03-11,4.087798,50.0,4.0,5.0,8.0,10.0
...,...,...,...,...,...,...,...,...
4495,STORE_03_FOODS_1_005,2023-12-27,4.912346,50.0,5.0,6.0,9.0,11.0
4496,STORE_03_FOODS_1_005,2023-12-28,5.007480,50.0,5.0,6.0,9.0,11.0
4497,STORE_03_FOODS_1_005,2023-12-29,5.007480,50.0,5.0,6.0,9.0,11.0
4498,STORE_03_FOODS_1_005,2023-12-30,5.007480,50.0,5.0,6.0,9.0,11.0


## Avaliação

In [304]:
import numpy as np
import pandas as pd

def pinball_loss(y_true, y_pred, quantile):
    """
    Calcula a Pinball Loss para um quantil específico (tau).
    """
    err = y_true - y_pred
    return np.maximum(quantile * err, (quantile - 1) * err).mean()

def avaliar_quantis_pipeline(df_saida_final, quantis=[0.50, 0.67, 0.95, 0.99]):
    """
    Etapa 4: Avalia a qualidade dos quantis gerados pela Binomial Negativa.
    """
    resultados = {}
    
    for q in quantis:
        col_q = f'q_{int(q*100)}'
        
        # 1. Calculando a Pinball Loss do quantil
        loss = pinball_loss(
            y_true=df_saida_final['y'], 
            y_pred=df_saida_final[col_q], 
            quantile=q
        )
        
        # 2. Checando a Cobertura Real (Empirical Coverage)
        # Em q=0.95, a venda real 'y' DEVE ser menor ou igual ao quantil 'q_95' em cerca de 95% dos dias.
        cobertura_real = (df_saida_final['y'] <= df_saida_final[col_q]).mean()
        
        resultados[f'q_{int(q*100)}'] = {
            'Pinball_Loss': round(loss, 4),
            'Cobertura_Alvo': f"{int(q*100)}%",
            'Cobertura_Empirica': f"{cobertura_real*100:.1f}%"
        }
        
    return pd.DataFrame(resultados).T

# --- Execução no Pipeline ---
# Avaliando os resultados que vieram do df_saida_final da Etapa 3/6
df_avaliacao = avaliar_quantis_pipeline(df_estruturado)

In [305]:
df_avaliacao

,Pinball_Loss,Cobertura_Alvo,Cobertura_Empirica
q_50,1.2628,50%,82.8%
q_67,1.1561,67%,90.0%
q_95,0.3629,95%,97.7%
q_99,0.1048,99%,99.3%
